# IAF approximation for phylogenetic VI

Explores inverse autoregressive flows (IAF) as a more flexible
variational family than mean-field or full-rank Gaussian.
Mean-field and full-rank runs are reproduced here for comparison.


## Setup


In [ ]:
import numpy as np
import yaml
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from tqdm import tqdm

tf.get_logger().setLevel("ERROR")
tfd = tfp.distributions

from treeflow import (
    Alignment,
    AlignmentFormat,
    parse_newick,
    PhyloModel,
    convert_tree_to_tensor,
)
from treeflow.model.phylo_model import phylo_model_to_joint_distribution
from treeflow.distributions import (
    DiscretizedDistribution,
    DiscreteParameterMixture,
    LeafCTMC,
)
from treeflow.distributions.tree import ConstantCoalescent
from treeflow.evolution.substitution import HKY, get_transition_probabilities_tree
from treeflow.vi import (
    fit_fixed_topology_variational_approximation,
    RobustOptimizer,
    RelativeLossNotDecreasing,
)

In [ ]:
NUM_RUNS = 5          # number of independent ADVI runs
NUM_STEPS = 50_000    # maximum steps (runs stop earlier if the criterion triggers)
LEARNING_RATE = 0.001
N_SAMPLES = 4_000     # posterior samples drawn per run for marginal comparison

# RelativeLossNotDecreasing convergence criterion:
# stop when EWMA(per-step decrease) / |ELBO| < CONVERGENCE_RTOL.
# This is invariant to starting conditions and dataset scale.
# CONVERGENCE_ATOL provides an optional absolute fallback.
CONVERGENCE_RTOL = 1e-6
CONVERGENCE_ATOL = None   # set to a float to add an absolute fallback
CONVERGENCE_WINDOW = 1_000
CONVERGENCE_MIN_STEPS = 5_000
CONVERGENCE_MIN_CONSECUTIVE = 3

In [ ]:
alignment = Alignment("../examples/demo-data/YFV.nex", format=AlignmentFormat.NEXUS)
starting_tree = parse_newick("../examples/demo-data/YFV.newick")
starting_tensor_tree = convert_tree_to_tensor(starting_tree)
sequence_tensor = alignment.get_encoded_sequence_tensor(starting_tree.taxon_set)
print("taxa:", starting_tree.taxon_count, "| sites:", alignment.site_count)

In [ ]:
model_string = """
clock:
  strict:
    clock_rate:
      lognormal:
        loc: -2.0
        scale: 2.0
site:
  discrete_gamma:
    category_count: 4
    site_gamma_shape:
      lognormal:
        loc: 0.0
        scale: 1.0
substitution:
  hky:
    kappa:
      lognormal:
        loc: 1.0
        scale: 1.25
    frequencies:
      dirichlet:
        concentration:
        - 2.0
        - 2.0
        - 2.0
        - 2.0
tree:
  coalescent:
    pop_size:
      lognormal:
        loc: 1.0
        scale: 1.5
"""

model = PhyloModel(yaml.safe_load(model_string))
base_model_dist = phylo_model_to_joint_distribution(
    model, starting_tensor_tree, alignment
)
prior_dists = {d.name: d for d in base_model_dist._get_single_sample_distributions()}
site_category_count = 4
subst_model = HKY()

In [ ]:
def build_model():
    """Build the pinned joint distribution with the native likelihood."""
    def alignment_dist(kappa, frequencies, tree, clock_rate, site_gamma_shape):
        distance_tree = tree.get_unrooted_tree() * clock_rate
        site_rate_distribution = DiscretizedDistribution(
            category_count=site_category_count,
            distribution=tfd.Gamma(site_gamma_shape, site_gamma_shape),
        )
        transition_probs_tree = get_transition_probabilities_tree(
            distance_tree,
            subst_model,
            rate_categories=site_rate_distribution.normalised_support,
            frequencies=frequencies,
            kappa=kappa,
        )
        leaf_ctmc = LeafCTMC(
            transition_probs_tree,
            tf.expand_dims(frequencies, -2),
            use_native=True,
        )
        site_mixture = DiscreteParameterMixture(site_rate_distribution, leaf_ctmc)
        return tfd.Sample(site_mixture, sample_shape=alignment.site_count)

    return tfd.JointDistributionNamed(
        dict(
            pop_size=prior_dists["pop_size"],
            kappa=prior_dists["kappa"],
            frequencies=prior_dists["frequencies"],
            site_gamma_shape=prior_dists["site_gamma_shape"],
            clock_rate=prior_dists["clock_rate"],
            tree=lambda pop_size: ConstantCoalescent(
                starting_tensor_tree.taxon_count,
                pop_size,
                starting_tensor_tree.sampling_times,
                tree_name="tree",
            ),
            alignment=alignment_dist,
        )
    ).experimental_pin(alignment=sequence_tensor)

In [ ]:
# Prior medians in constrained space (exp(loc) for lognormals, uniform simplex
# for frequencies).  Used as a deterministic starting point so that run-to-run
# differences are due only to ADVI sampling noise, not initialization luck.
INIT_LOC = dict(
    tree=starting_tensor_tree,
    clock_rate=tf.constant(np.exp(-2.0), dtype=tf.float64),       # lognormal(loc=-2, scale=2) median
    kappa=tf.constant(np.exp(1.0), dtype=tf.float64),              # lognormal(loc=1, scale=1.25) median
    pop_size=tf.constant(np.exp(1.0), dtype=tf.float64),           # lognormal(loc=1, scale=1.5) median
    site_gamma_shape=tf.constant(np.exp(0.0), dtype=tf.float64),   # lognormal(loc=0, scale=1) median
    frequencies=tf.constant([0.25, 0.25, 0.25, 0.25], dtype=tf.float64),
)

convergence_criterion = RelativeLossNotDecreasing(
    rtol=CONVERGENCE_RTOL,
    atol=CONVERGENCE_ATOL,
    window_size=CONVERGENCE_WINDOW,
    min_num_steps=CONVERGENCE_MIN_STEPS,
    min_consecutive=CONVERGENCE_MIN_CONSECUTIVE
)


def fit(seed):
    pinned = build_model()
    optimizer = RobustOptimizer(tf.optimizers.Adam(learning_rate=LEARNING_RATE))
    approx, trace = fit_fixed_topology_variational_approximation(
        pinned,
        topologies=dict(tree=starting_tree.topology),
        optimizer=optimizer,
        num_steps=NUM_STEPS,
        convergence_criterion=convergence_criterion,
        return_full_length_trace=False,
        init_loc=INIT_LOC,
        progress_bar=tqdm,
        seed=seed,
    )
    loss = np.asarray(trace.loss)
    param_traces = {
        name: np.asarray(var) for name, var in trace.parameters.items()
    }
    cc_state = trace.convergence_criterion_state
    ewma = np.asarray(cc_state.average_decrease)
    rel_rate = np.asarray(cc_state.rel_rate)
    return approx, loss, param_traces, ewma, rel_rate

## Mean-field reference runs


In [ ]:
seeds = [tf.constant([i, i], dtype=tf.int32) for i in range(1, NUM_RUNS + 1)]

run_results = []
for i, seed in enumerate(seeds):
    print(f'MF run {i+1}/{NUM_RUNS}  seed={seed.numpy().tolist()}')
    approx, loss, ewma, rel_rate = fit(seed)
    run_results.append(dict(approx=approx, loss=loss, ewma=ewma, rel_rate=rel_rate))
    print(f'  {len(loss)} steps, ELBO={-loss[-1]:.2f}')

all_samples = [
    r['approx'].sample(N_SAMPLES, seed=sample_seed) for r in run_results
]


## Full-rank reference runs


In [ ]:
from treeflow.model.approximation import get_fixed_topology_full_rank_approximation

NUM_FR_RUNS = 3


def fit_full_rank(seed):
    pinned = build_model()
    optimizer = RobustOptimizer(tf.optimizers.Adam(learning_rate=LEARNING_RATE))
    approx, trace = fit_fixed_topology_variational_approximation(
        pinned,
        topologies=dict(tree=starting_tree.topology),
        optimizer=optimizer,
        num_steps=NUM_STEPS,
        convergence_criterion=convergence_criterion,
        return_full_length_trace=False,
        init_loc=INIT_LOC,
        seed=seed,
        approx_fn=get_fixed_topology_full_rank_approximation,
    )
    loss = np.asarray(trace.loss)
    cc_state = trace.convergence_criterion_state
    return approx, loss, np.asarray(cc_state.average_decrease), np.asarray(cc_state.rel_rate)


fr_seeds = [tf.constant([i + 10, i + 10], dtype=tf.int32) for i in range(1, NUM_FR_RUNS + 1)]

fr_results = []
for i, seed in enumerate(fr_seeds):
    print(f'FR run {i+1}/{NUM_FR_RUNS}  seed={seed.numpy().tolist()}')
    approx, loss, ewma, rel_rate = fit_full_rank(seed)
    fr_results.append(dict(approx=approx, loss=loss, ewma=ewma, rel_rate=rel_rate))
    print(f'  {len(loss)} steps, ELBO={-loss[-1]:.2f}')

fr_samples = [
    r['approx'].sample(N_SAMPLES, seed=sample_seed) for r in fr_results
]


In [ ]:
# Variables needed by the three-way comparison plot below
_fr_clock  = np.concatenate([marginal(s, 'clock_rate')  for s in fr_samples])
_fr_height = np.concatenate([marginal(s, 'root_height') for s in fr_samples])


## IAF (normalising-flow) approximation

If the posterior is curved in log–log space, a more flexible
approximation family might close the remaining gap.  We use *inverse
autoregressive flow* (IAF): a stack of masked autoregressive networks
that warps a Gaussian base into an arbitrarily flexible distribution.

**Two warm-starting tricks** to stabilise IAF on this dataset:

1. **Affine base** — the IAF chain includes trainable `loc` and `log_scale`
   variables before the autoregressive bijectors.  With the network weights
   at zero (identity flows), the distribution is `Normal(loc, softplus(log_scale))`
   — a mean-field approximation.  `init_loc` seeds `loc` from the prior medians.

2. **Surrogate warm-up** — before ELBO optimisation we run 2 000 steps minimising
   `-E_{q_{IAF}}[log q_{MF}(z)]`, which aligns the IAF's distribution with the
   already-fitted mean-field approximation (cheap: no likelihood evaluations).
   Gradients are clipped to 5.0 to prevent early NaN explosions.
   ELBO optimisation then starts from a sensible initial approximation rather
   than a random scramble.


In [ ]:
from functools import partial
from treeflow.model.approximation import get_fixed_topology_inverse_autoregressive_flow_approximation
from treeflow.vi.util import default_vi_trace_fn
from tensorflow_probability.python.vi import fit_surrogate_posterior
from tensorflow_probability.python.math.minimize import (
    _truncate_at_has_converged,
    _trace_has_converged,
)

NUM_IAF_RUNS = 3
NUM_IAF_WARMUP_STEPS = 2000
NUM_IAF_WARMUP_SAMPLES = 32

# Use the first (already fitted) MF run as the warm-up target
mf_warmup_target = run_results[0]["approx"]


def fit_iaf(seed):
    pinned = build_model()

    # Build IAF — now includes trainable affine base initialised at prior medians
    iaf_approx, iaf_vars = get_fixed_topology_inverse_autoregressive_flow_approximation(
        pinned,
        topology_pins=dict(tree=starting_tree.topology),
        hidden_units_per_layer=starting_tree.taxon_count,
        init_loc=INIT_LOC,
    )
    trainable_vars = list(iaf_vars.values())

    # --- Surrogate warm-up ---
    # Minimise -E_{q_IAF}[log q_MF(z)] to align IAF with MF before touching the likelihood.
    # Gradient clipping prevents NaN explosions in the autoregressive network.
    warmup_opt = tf.optimizers.Adam(learning_rate=LEARNING_RATE)

    @tf.function
    def warmup_step():
        with tf.GradientTape() as tape:
            sample = iaf_approx.sample(NUM_IAF_WARMUP_SAMPLES)
            loss = -tf.reduce_mean(mf_warmup_target.log_prob(sample))
        grads = tape.gradient(loss, trainable_vars)
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        warmup_opt.apply_gradients(zip(grads, trainable_vars))
        return loss

    print(f"  warm-up ({NUM_IAF_WARMUP_STEPS} steps)...", flush=True)
    for step in range(NUM_IAF_WARMUP_STEPS):
        warmup_step()
    print(f"  warm-up done", flush=True)

    # --- Full ELBO optimisation ---
    elbo_opt = RobustOptimizer(tf.optimizers.Adam(learning_rate=LEARNING_RATE))
    trace_fn = partial(default_vi_trace_fn, variables_dict=iaf_vars)
    aug_trace_fn = _trace_has_converged(trace_fn, tf.reduce_all)

    trace = fit_surrogate_posterior(
        pinned.unnormalized_log_prob,
        iaf_approx,
        elbo_opt,
        NUM_STEPS,
        convergence_criterion=convergence_criterion,
        trace_fn=aug_trace_fn,
        seed=seed,
    )
    trace = _truncate_at_has_converged(trace)
    loss = np.asarray(trace.loss)
    cc_state = trace.convergence_criterion_state
    return iaf_approx, loss, np.asarray(cc_state.average_decrease), np.asarray(cc_state.rel_rate)


iaf_seeds = [tf.constant([i + 20, i + 20], dtype=tf.int32) for i in range(1, NUM_IAF_RUNS + 1)]

iaf_results = []
for i, seed in enumerate(iaf_seeds):
    print(f"\n--- IAF run {i + 1} / {NUM_IAF_RUNS}  (seed={seed.numpy().tolist()}) ---")
    try:
        approx, loss, ewma, rel_rate = fit_iaf(seed)
        n = len(loss)
        converged = n < NUM_STEPS
        print(f"  {'converged' if converged else 'DID NOT CONVERGE'}: {n} steps, ELBO = {-loss[-1]:.2f}")
        iaf_results.append(dict(approx=approx, loss=loss, ewma=ewma, rel_rate=rel_rate,
                                failed=False))
    except Exception as e:
        print(f"  FAILED: {e}")
        iaf_results.append(dict(approx=None, loss=None, ewma=None, rel_rate=None, failed=True))

n_ok = sum(1 for r in iaf_results if not r["failed"])
print(f"\n{n_ok}/{NUM_IAF_RUNS} IAF runs completed without error.")


In [ ]:
iaf_ok = [r for r in iaf_results if not r["failed"]]
iaf_samples = [r["approx"].sample(N_SAMPLES, seed=sample_seed) for r in iaf_ok]

if iaf_samples:
    iaf_rows = []
    for name in POSTERIOR_PARAMS:
        for i, samples in enumerate(iaf_samples):
            vals = marginal(samples, name)
            iaf_rows.append(dict(parameter=name, run=i + 1, mean=vals.mean(), std=vals.std()))

    df_iaf = pd.DataFrame(iaf_rows)
    iaf_stability = df_iaf.groupby("parameter").agg(
        mean_of_means=("mean", "mean"),
        std_of_means=("mean", "std"),
        mean_post_sd=("std", "mean"),
    )
    iaf_stability["inter_run / post_sd"] = (
        iaf_stability["std_of_means"] / iaf_stability["mean_post_sd"]
    ).abs()
    print(f"IAF stability ({len(iaf_ok)} successful runs):")
    display(iaf_stability.round(4))
else:
    print("No successful IAF runs to analyse.")

# --- Three-way ELBO comparison ---
print("\nELBO comparison (final step):")
for label, results in [("Mean-field", run_results), ("Full-rank", fr_results)]:
    elbos = [-r["loss"][-1] for r in results]
    print(f"  {label:12s}: {np.mean(elbos):.1f} ± {np.std(elbos):.1f}  "
          f"(range {min(elbos):.1f} – {max(elbos):.1f})")
if iaf_ok:
    elbos = [-r["loss"][-1] for r in iaf_ok]
    print(f"  {'IAF':12s}: {np.mean(elbos):.1f} ± {np.std(elbos):.1f}  "
          f"(range {min(elbos):.1f} – {max(elbos):.1f}, {len(iaf_ok)}/{NUM_IAF_RUNS} succeeded)")
else:
    print(f"  {'IAF':12s}: all runs failed (NaN gradient instability)")

In [ ]:
fig, axs = plt.subplots(1, len(POSTERIOR_PARAMS), figsize=(3.5 * len(POSTERIOR_PARAMS), 3.5))

approx_configs = [
    ("mean-field", all_samples,  "steelblue",   0.20),
    ("full-rank",  fr_samples,   "darkorange",   0.30),
]
if iaf_samples:
    approx_configs.append(("IAF", iaf_samples, "forestgreen", 0.35))

for ax, name in zip(axs, POSTERIOR_PARAMS):
    for label, samples_list, color, alpha in approx_configs:
        for i, samples in enumerate(samples_list):
            vals = marginal(samples, name)
            ax.hist(vals, bins=40, alpha=alpha, density=True, color=color,
                    label=label if i == 0 else None)
    ax.set_title(name)
    ax.set_yticks([])

axs[0].legend(fontsize=9)
labels_str = f"{NUM_RUNS} MF, {NUM_FR_RUNS} FR" + (f", {len(iaf_ok)} IAF" if iaf_ok else ", IAF failed")
fig.suptitle(f"Posterior marginals: MF (blue) vs FR (orange)" +
             (" vs IAF (green)" if iaf_ok else " — IAF unstable") +
             f"\n({labels_str} runs pooled)")
fig.tight_layout()
plt.show()

# IAF geometry if available
if iaf_samples:
    _iaf_clock = np.concatenate([marginal(s, "clock_rate") for s in iaf_samples])
    _iaf_height = np.concatenate([marginal(s, "root_height") for s in iaf_samples])

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (clk, ht, label, cmap) in zip(axs, [
        (_fr_clock,  _fr_height,  "Full-rank", "Blues"),
        (_iaf_clock, _iaf_height, "IAF",       "Greens"),
    ]):
        ax.hist2d(np.log(clk), np.log(ht), bins=50, cmap=cmap, density=True)
        c = np.polyfit(np.log(clk), np.log(ht), 1)
        x = np.linspace(np.log(clk).min(), np.log(clk).max(), 100)
        ax.plot(x, np.polyval(c, x), "r-", lw=1.5, label=f"slope={c[0]:.2f}")
        ax.set_xlabel("log clock_rate"); ax.set_ylabel("log root_height")
        ax.set_title(f"{label}: log–log density"); ax.legend(fontsize=8)
    fig.suptitle("Clock–height ridge geometry: full-rank vs IAF")
    fig.tight_layout()
    plt.show()